# Supporting calculations

Each cell reproduces one estimate in the proposal. Inputs are cited or marked *assumed*.

| Cell | Proposal claim | Key sources |
|---|---|---|
| 1 | Float tops 87 vs 50 °C; 31 vs 11 W/m² into water (July) | Duffie & Beckman, *Solar Eng. of Thermal Processes*, 5th ed. (2020), Ch. 3 & 6, [doi](https://doi.org/10.1002/9781119540328); Nellis AFB TMY3 |
| 2 | Box 47 kg, 14 cm draft, 6 cm freeboard (Fig. 3) | module mass: JinkoSolar datasheet; 1.5 mm gauge: Trico; 0.8 m: furnace width |
| 3 | 7,200–7,500 m²/MW; 13,500–15,600 m³/yr water saved | module: JinkoSolar datasheet; evaporation: USGS SIR 2013-5229 |
| 4 | ≤ ~190 kg CO₂e/kWp | IEA-PVPS T12-29:2024 |
| 5 | Floats 19–26 % of cost; break-even 1.5–1.9× | NREL/TP-7A40-80695 (2021); gains from the Fig. 2 ray trace |


In [1]:
import os, urllib.request
os.makedirs("data", exist_ok=True)  # fetch source data when run outside the repo (e.g. Colab)
for f in ['USA_NV_Nellis.AFB.723865_TMY3.epw']:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/fpv-enamel/enamel-float-notebooks/main/calcs/data/" + f, "data/" + f)

# 1. Float energy balance (hourly, July): absorbed sun + sky IR = emitted IR + wind convection + conduction to water
import numpy as np
from scipy.optimize import brentq
!pip -q install pvlib
import pvlib

wx = pvlib.iotools.read_epw("data/USA_NV_Nellis.AFB.723865_TMY3.epw")[0]
jul = wx[wx.index.month == 7]
SIG, U, Tw = 5.67e-8, 2.0, 28.0          # U: top-to-water conductance, assumed; Tw: July water temp, assumed
h = lambda v: 5.7 + 3.8 * v              # wind convection, D&B Eq. 3.15.2 (incl. radiation -> conservative)

def float_top(a, eps):
    """Hourly top temperature [C] and heat into water [W/m2] for solar absorptance a, emissivity eps."""
    T, Q = [], []
    for _, r in jul.iterrows():
        f = lambda Ts: (a * r.ghi + eps * r.ghi_infrared - eps * SIG * Ts**4
                        - h(r.wind_speed) * (Ts - r.temp_air - 273.15) - U * (Ts - Tw - 273.15))
        Ts = brentq(f, 200, 400)
        T.append(Ts - 273.15); Q.append(U * (Ts - Tw - 273.15))
    return max(T), np.mean(Q)

for name, a in (("black HDPE (a 0.95, assumed)", 0.95), ("white enamel (a 0.21, Leggett & King)", 0.21)):
    Tmax, Q = float_top(a, 0.90)
    print(f"{name:38s} max top {Tmax:.0f} C, mean heat into water {Q:.0f} W/m2")


black HDPE (a 0.95, assumed)           max top 87 C, mean heat into water 31 W/m2
white enamel (a 0.21, Leggett & King)  max top 50 C, mean heat into water 11 W/m2


In [2]:
# 2. Buoyancy of the sealed box (Archimedes): draft = total mass / (water density x footprint)
L, W, H = 1.6, 0.8, 0.20                       # m; W = furnace channel width, L and H assumed
t_s, rho_s = 1.5e-3, 7850                      # steel gauge (Trico), density
t_e, rho_e = 0.3e-3, 2500                      # enamel per side, assumed
area = 2 * (L * W + L * H + W * H)
box = area * (t_s * rho_s + 2 * t_e * rho_e)
load = 32.4 + 100                              # module (datasheet) + worker (assumed)
draft = (box + load) / (1000 * L * W)
print(f"box {box:.1f} kg, draft {100 * draft:.1f} cm, freeboard {100 * (H - draft):.1f} cm")


box 46.7 kg, draft 14.0 cm, freeboard 6.0 cm


In [3]:
# 3. Array area and water saved per MW
area_mod, P = 2.382 * 1.134, np.array([600, 625])        # m2, W (datasheet)
A = 1e6 / (0.6 * P / area_mod)                           # m2 of water per MW at GCR 0.6
print("area per MW:", A.round(-2)[::-1], "m2")
print("water saved:", (A[::-1] * np.array([1.881, 2.074])).round(-2), "m3/yr (if evaporation stops entirely)")


area per MW: [7200. 7500.] m2
water saved: [13500. 15600.] m3/yr (if evaporation stops entirely)


In [4]:
# 4. Carbon of polyethylene floats, cradle-to-gate
print(f"{44 * 2.9:.0f}-{44 * 4.3:.0f} kg CO2e/kWp  (44 kg HDPE/kWp x 2.9-4.3 kg CO2e/kg)")


128-189 kg CO2e/kWp  (44 kg HDPE/kWp x 2.9-4.3 kg CO2e/kg)


In [5]:
# 5. Float cost share s and break-even multiple m = 1 + g/s (capital cost per kWh unchanged)
s = np.array([0.25 * 0.75, 0.30 * 0.85])        # structural BOS 25-30 % of cost, floats 75-85 % of it (NREL)
g = np.array([0.14, 0.16])                      # array energy gain, white vs black floats (Fig. 2 ray trace: 13.9-16.3 %)
print(f"float share {100 * s[0]:.0f}-{100 * s[1]:.0f} %;  break-even {1 + g[0] / s[1]:.2f}-{1 + g[1] / s[0]:.2f} x")


float share 19-26 %;  break-even 1.55-1.85 x
